<a href="https://colab.research.google.com/github/addadugurudurga2024-lang/Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit



## 1. Two paper findings + my methodology questions


### Finding 1 — High Search Volume = More Traffic

The FlyRank research paper reports that stored search volume had only a weak relationship with page impressions. The paper therefore treats search volume as a competition hint rather than a literal traffic forecast.

**Methodology question:**  
This is an observational relationship, so I would ask whether the result remains consistent across different clients, content types, and time periods. I would also ask whether ranking position, search intent, or other page-level factors could explain part of the relationship.

I would treat this finding as directional rather than causal. The evidence supports an observed relationship in the analyzed dataset, but it does not establish that search volume causes traffic.

### Finding 2 — Content With Flags Is Failing

The paper reports that pages with 2–3 optimization flags could have higher health scores than pages with zero flags. The interpretation is that flags can indicate sufficient visibility or data to diagnose an issue, rather than proving that the content itself is weak.

**Methodology question:**  
I would ask how the optimization-flag labels are generated and whether pages need sufficient impressions or behavioral data before some flags can appear. I would also ask whether page age and visibility were controlled when comparing pages with different numbers of flags.

I would therefore interpret flags as workflow-priority signals rather than direct evidence that a page is failing.

## 2. My model under an honest split (before/after)

The Week-5 model used a standard random 80/20 train-test split and achieved 0.637 accuracy, with precision of 0.644, recall of 0.738, and F1 score of 0.688.

For this validation audit, I use a grouped-by-client split. The same client is kept entirely within either the training or testing group, preventing observations from the same client from appearing on both sides of the validation split.

This is a more conservative validation design for the content opportunity scoring question because it tests whether the model can generalize to clients that were not represented in training.

The same target, features, and Decision Tree configuration from ML-08 are retained. Only the validation design is changed.

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Create target
df["is_declining"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

# Same features as ML-08
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df["is_declining"]

# Client groups
groups = df["client_id"]

# Grouped train/test split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Same Decision Tree as ML-08
model_grouped = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model_grouped.fit(X_train, y_train)

pred_grouped = model_grouped.predict(X_test)

# Metrics
grouped_accuracy = accuracy_score(y_test, pred_grouped)
grouped_precision = precision_score(
    y_test, pred_grouped, zero_division=0
)
grouped_recall = recall_score(
    y_test, pred_grouped, zero_division=0
)
grouped_f1 = f1_score(
    y_test, pred_grouped, zero_division=0
)

comparison = pd.DataFrame({
    "Validation": [
        "ML-08 Random Split",
        "ML-09 Grouped by Client"
    ],
    "Accuracy": [
        0.637,
        round(grouped_accuracy, 3)
    ],
    "Precision": [
        0.644,
        round(grouped_precision, 3)
    ],
    "Recall": [
        0.738,
        round(grouped_recall, 3)
    ],
    "F1": [
        0.688,
        round(grouped_f1, 3)
    ]
})

display(comparison)

print("Grouped Accuracy :", round(grouped_accuracy, 3))
print("Grouped Precision:", round(grouped_precision, 3))
print("Grouped Recall   :", round(grouped_recall, 3))
print("Grouped F1       :", round(grouped_f1, 3))

,Validation,Accuracy,Precision,Recall,F1
0,ML-08 Random Split,0.637,0.644,0.738,0.688
1,ML-09 Grouped by Client,0.557,0.565,0.581,0.573


Grouped Accuracy : 0.557
Grouped Precision: 0.565
Grouped Recall   : 0.581
Grouped F1       : 0.573


In [8]:
# Create error analysis dataframe
test_results = df.iloc[test_idx].copy()

test_results["actual"] = y_test.values
test_results["prediction"] = pred_grouped

test_results["correct"] = (
    test_results["actual"] == test_results["prediction"]
)

# Show actual model failures
failure_examples = test_results[
    ~test_results["correct"]
][[
    "actual",
    "prediction",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]].head(10)

display(failure_examples)

,actual,prediction,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count
1,1,0,445,25,15320,20.3,0.05,2481.0
13,0,1,238,103,307,39.8,0.00,1342.0
19,1,0,187,20,99,6.9,2.02,2673.0
23,1,0,502,20,297,13.9,0.34,NaN
26,0,1,300,13,2426,30.0,0.12,2686.0
37,1,0,228,13,187,11.5,1.07,2658.0
39,1,0,348,104,4,36.3,0.00,3666.0
51,1,0,126,8,2,7.5,0.00,2756.0
54,1,0,502,22,170,10.7,0.00,NaN
56,0,1,145,20,16,4.6,0.00,3158.0


### Error analysis

The failure examples show cases where the model's predicted class differs from the observed declining label.

These errors indicate that the selected page-level features do not perfectly separate declining and non-declining pages. Pages with similar age, visibility, position, CTR, and content characteristics can still have different observed outcomes.

The examples are used for diagnostic purposes. They do not establish a causal reason for why an individual page declined.

The model is therefore better interpreted as directional decision-support for prioritization rather than as a definitive classifier of which pages require updates.

## 3. Leakage audit

I reviewed the final feature set for direct and temporal leakage.

| Feature | Leakage assessment |
|---|---|
| `content_age_days` | Describes page age and does not directly contain the target. |
| `days_since_last_update` | Describes update recency and does not directly contain the target. |
| `impressions_90d` | Must represent information available before the prediction point. |
| `avg_position` | Must come only from the historical feature window and not the future outcome period. |
| `ctr` | Must be calculated only from information available before prediction. |
| `word_count` | Describes content characteristics and does not directly contain the target. |

The main leakage risk is temporal leakage rather than an explicit target-containing feature. Performance features such as impressions, average position, and CTR are acceptable only when they are calculated from information available before the prediction period.

The grouped-by-client split prevents the same client from appearing in both training and testing groups, but it does not by itself guarantee that every feature is temporally clean. The feature-window construction must therefore remain an important condition of the model's validity.

## 4. Claim rewrite

### Original claim

"The Decision Tree can identify webpages that need content updates."

### Safer claim

"On the evaluated dataset, the Decision Tree showed measured predictive performance for the defined declining-page label. The grouped-by-client validation provides directional evidence about whether the selected features can support prioritization of pages for review. The model should be treated as decision-support rather than as an automatic determination that a page requires an update."

### Before/after interpretation

The random split produced an accuracy of 0.637, while the grouped-by-client result above provides a more conservative estimate of performance on unseen clients.

The difference between the two validation designs is itself an important finding. It shows why validation design matters when estimating how a model may generalize beyond the data used during development.

These results are interpreted as measured validation results on this dataset rather than as evidence of universal model performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [✅  ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅  ] No client names, URLs, or private queries anywhere
- [✅  ] My claims use careful words: observed, measured, directional, decision-support
- [✅  ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.